In [17]:
import os
import time
import pandas as pd
import io
import random
import numpy as np
from tqdm import tqdm
from groq import Groq
from dotenv import load_dotenv

def set_seed(seed: int):
    random.seed(seed) # Python
    np.random.seed(seed)  # Numpy, é o gerador utilizado pelo sklearn
    os.environ['PYTHONHASHSEED'] = str(seed)  # sistema operativo

set_seed(25)
load_dotenv()

True

In [18]:
# Initialize Groq client
client = Groq(api_key=os.getenv('GROQ_API_KEY'))
model='gemma2-9b-it'
batch_size=10
df = pd.read_csv('dataset2_disclosed_complete_inputs.csv', sep='\t')
output_path = 'submissao3-grupo001-s1.csv'


In [19]:
classification_prompt = (
    'You are an expert in distinguishing AI-generated text from Human written text.\n'
    'Analyze each provided text sample and classify it strictly as either "AI" or "Human".\n'
    '\n'
    'Classification Guidelines:\n'
    'AI:\n'
    '- More uniform in structure and style\n'
    '- Lacks emotional depth or personal experience\n'
    '- Often formal, technical, and predictable\n'
    '\n'
    'Human:\n'
    '- Includes emotions, opinions, or personal experiences\n'
    '- Displays irregularities and diverse sentence structures\n'
    '- Uses idioms, slang, or informal language\n'
    'Note that I need the format to be a CSV with "ID" and "Label", and nothing else.\n'
    'Here are some classified examples:\n'

"Using entanglement of assistance, we establish a general polygamy inequality of multi-party entanglement in arbitrary dimensional quantum systems. For multi-party closed quantum systems, we relate our result with the monogamy of entanglement to show that the entropy of entanglement is an universal entanglement measure that bounds both monogamy and polygamy of multi-party quantum entanglement.,Human"
"The verification of quantum entanglement under the influence of realistic noise and decoherence is crucial for the development of quantum technologies. Unfortunately, a full entanglement characterization is generally not possible with most entanglement criteria such as entanglement witnesses or the partial transposition criterion. In particular, so called bound entanglement cannot be certified via the partial transposition criterion. Here we present the full entanglement verification of dephased qubit and qutrit Werner states via entanglement quasiprobabilities. Remarkably, we are able to reveal bound entanglement for noisy-mixed states in,Human"
"This paper discusses relationships between topological entanglement and quantum entanglement. Specifically, we propose that for this comparison it is fundamental to view topological entanglements such as braids as entanglement operators and to associate to them unitary operators that are capable of creating quantum entanglement.,Human"
"We study maximally entangled states and fully entangled fraction in general d'\otimes d (d'\geq d) systems. Necessary and sufficient conditions for maximally entangled pure and mixed states are presented. As a natural generalization of the usual fully entangled fraction for d\otimes d systems, we define the maximal overlap between a given quantum state and the maximally entangled states as the fully entangled fraction in d'\otimes d systems. The properties of this fully entangled fraction and its relations to quantum teleportation have been analyzed. The witness for detecting maximally entangled states and quantum states that are useful for quantum teleportation is provided.,Human"
"We study entanglement properties of all eigenstates of the Heisenberg XXX model, and find that the entanglement and mixedness for a pair of nearest-neighbor qubits are completely determined by the corresponding eigenenergies. Specifically, the negativity of the eigenenergy implies pairwise entanglement. From the relation between entanglement and eigenenergy, we obtain finite-size behaviors of the entanglement. We also study entanglement and mixedness versus energy in the quantum Heisenberg XY model.,Human"
"We have found a quantum cloning machine that optimally duplicates the entanglement of a pair of d-dimensional quantum systems. It maximizes the entanglement of formation contained in the two copies of any maximally-entangled input state, while preserving the separability of unentangled input states. Moreover, it cannot increase the entanglement of formation of all isotropic states. For large $d$, the entanglement of formation of each clone tends to one half the entanglement of the input state, which corresponds to a classical behavior. Finally, we investigate a local entanglement cloner, which yields entangled clones with one fourth the input entanglement,Human"
"In this paper based on the notion of entanglement witness, a new measure of entanglement called floating entanglement witness measure is introduced which satisfies some of the usual properties of a good entanglement measure. By exploiting genetic algorithm, we introduce a classical algorithm that computes floating entanglement witness measure. This algorithm also provides a method for finding entanglement witness for a given entangled state.,Human"
"We discuss maximum entangled states of quantum systems in terms of quantum fluctuations of all essential measurements responsible for manifestation of entanglement. Namely, we consider maximum entanglement as a property of states, for which quantum fluctuations come to their extreme.,Human"
"Entanglement is one of the essential resources in quantum information and communication technology. The entanglement thus far explored and applied to QICT has been pure and distillable entanglement. Yet there is another type of entanglement, called 'bound entanglement', which is not distillable by local operations and classical communication (LOCC). We demonstrate the experimental 'activation' of the bound entanglement held in the four-qubit Smolin state, unleashing its immanent entanglement in distillable form, with the help of auxiliary two-qubit entanglement and LOCC. We anticipate that it opens the way to a new class of QICT applications,Human"
"Quantum entanglement swapping is one of the most promising ways to realize the quantum connection among local quantum nodes. In this Letter, we present an experimental demonstration of the entanglement swapping between two independent multipartite entangled states, each of which involves a tripartite Greenberger-Horne-Zeilinger (GHZ) entangled state of an optical field. The entanglement swapping is implemented deterministically by means of a joint measurement on two optical modes coming from the two multipartite entangled states respectively and the classical feedforward of the measurement results. After entanglement swapping the two independent multipartite entangled states are merged into a large entangled state in which all unmeasured quantum modes are entangled. The entanglement swapping between a tripartite GHZ state and an Einstein-Podolsky-Rosen entangled state is,Human"
"Entanglement measures quantify the amount of quantum entanglement that is contained in quantum states. Typically, different entanglement measures do not have to be partially ordered. The presence of a definite partial order between two entanglement measures for all quantum states, however, allows for meaningful conceptualization of sensitivity to entanglement, which will be greater for the entanglement measure that produces the larger numerical values. Here, we have investigated the partial order between the normalized versions of four entanglement measures based on Schmidt decomposition of bipartite pure quantum,Human"
"The entanglement measure for multiqudits is proposed. This measure calculates the partial entanglement distributed by subsystems and the complete entanglement of the total system. This shows that we need to measure the subsystem entanglements to explain the full description for multiqudit entanglement. Furthermore, we extend the entanglement measure to mixed multiqubits and the higher dimension Hilbert spaces.,Human"
"We consider composability of quantum channels from a limited amount of entanglement via local operations and classical communication. We show that any $k$-partially entanglement breaking channel can be composed from an entangled state with Schmidt number of $k$ via one-way LOCC. From the entanglement assisted construction we can reach an alternative definition of partially entanglement breaking channels.,Human"
"Many previous works on quantum photolithography are based on maximally-entangled states. In this paper, we generalize the MES quantum photolithography to the case where two light beams share a $N$-photon nonmaximally-entangled state. we investigate the correlations between quantum entanglement and quantum photolithography. It is shown that for nonlocal entanglement between the two light beams the amplitude of the deposition rate can be changed through varying the degree of entanglement described by an entanglement angle while the resolution remains unchanged, and found that for local entanglement between the two light beams the effective Rayleigh resolution of quantum photolithography can be resonantly enhanced.,Human"
"Quantum entanglement plays crucial roles in quantum information processing. Quantum entangled states have become the key ingredient in the rapidly expanding field of quantum information science. Although the nonclassical nature of entanglement has been recognized for many years, considerable efforts have been taken to understand and characterize its properties recently. In this review, we introduce some recent results in the theory of quantum entanglement. In particular separability criteria based on the Bloch representation, covariance matrix, normal form and entanglement witness; lower bounds, subadditivity property of concurrence and tangle; fully entangled fraction related to the optimal fidelity of quantum teleportation and entanglement distillation will be discussed in detail.,Human"

"Entanglement fidelity measures how well entanglement is preserved through quantum channels. Various measures of entanglement like entanglement of formation, concurrence, negativity quantify entanglement in a system. We explore the link between entanglement fidelity and these measures. We propose new fidelities based on these measures and compare them statistically with the existing entanglement fidelity.,ai"
"Quantum Entanglement for Internet: Key to the quantum internet's structure, entanglement lets us connect quantum nodes. We've created a way to manage entanglement access. This method assigns different entanglement access levels to users. The cost of each path depends on the quality of entanglement and the likelihood of entangled connections. We've found that more available entangled paths result in better entanglement quality and reliability. This scheme offers an efficient way to control entanglement access in the experimental quantum internet.,ai"
"Classical systems can exhibit entanglement, a phenomenon defined by coincidental correlations. A mechanical system, with a single conserved variable, can mimic quantum entanglement experiments with 77.8% conditional efficiency. This was demonstrated with four-particle entanglement swapping and GHZ entanglement.,ai"
"We've proven a general rule for entanglement in multi-party quantum systems of any dimension. For closed systems, we've shown that entanglement entropy is a universal measure that limits both monogamy and polygamy of multi-party entanglement.,ai"
"Verifying quantum entanglement in realistic, noisy conditions is vital for advancing quantum tech. Most entanglement tests, like witnesses or partial transposition, can't fully identify entanglement, especially 'bound entanglement'. We've developed a method using 'entanglement quasiprobabilities' to fully verify entanglement in dephased qubit and qutrit Werner states. Notably, we've detected bound entanglement in noisy, mixed states.,ai"
"The paper explores connections between topological and quantum entanglement. It suggests treating topological entanglements like braids as entanglement operators and pairing them with quantum entanglement-creating unitary operators.,ai"
"We investigate maximally entangled states and fully entangled fractions in general d'\otimes d (d'≥d) systems. We provide necessary and sufficient conditions for these states, both pure and mixed. We extend the usual fully entangled fraction for d\otimes d systems by defining it as the maximum overlap between a given state and maximally entangled states in d'\otimes d systems. We analyze the properties of this fraction and its connection to quantum teleportation. We also offer a method to detect,ai"
"Key challenges in creating a quantum internet are maintaining stable quantum repeaters and maximizing entanglement rates in complex networks. A stable quantum repeater ensures that all incoming quantum states can be replaced with a target state. Strong stability means reliable entanglement transfer with limited memory storage and delays. A new theoretical approach combines noise analysis and entanglement rate optimization. We introduce the 'entanglement swapping set' to represent a repeater's quantum memory status with stored states. We then find the best entanglement distribution strategy.,ai"
"We investigate the entanglement of all Heisenberg model eigenstates. Nearest-neighbor qubits' entanglement and mixedness are fully defined by their eigenenergies, with negative eigenenergies indicating pairwise entanglement. We derive finite-size entanglement behaviors from this relationship. Additionally, we study entanglement and mixedness changes with energy in the quantum Heisenberg XY model.,ai"
"We've created a quantum copier that perfectly duplicates the entanglement of pairs of $d$-dimensional quantum systems. It boosts entanglement in maximally-entangled inputs while keeping unentangled inputs separate. It doesn't increase entanglement in all cases. For large systems, each clone has about half the entanglement of the input, behaving classically. Lastly, we've tested a local copier that makes clones with one fourth the input's entanglement.,ai"
"This paper presents a new entanglement measure, the floating entanglement witness measure, based on entanglement witnesses. This measure meets some key criteria for a good entanglement measure. The authors also introduce a classical algorithm using genetic algorithms to compute this measure and identify entanglement witnesses for given entangled states.,ai"
"We explore the most entangled states in quantum systems by examining the fluctuations in crucial measurements that reveal entanglement. We define maximum entanglement as the state where these fluctuations reach their peak.,ai"
"Entanglement is crucial in quantum information and communication. Most used entanglement is pure and distillable. However, there's bound entanglement, which can't be converted into distillable form by local operations and classical communication. We've shown that bound entanglement in a four-qubit Smolin state can be activated, turning it into usable, distillable form using auxiliary entanglement and local operations. This could lead to new quantum tech applications.,ai"
"We've shown a way to connect two independent, complex quantum states. Each state was a GHZ state, a special type of entanglement involving three particles of light. We combined these states using a technique called 'entanglement swapping'. This involves measuring two light beams, one from each state, and then using the results of that measurement to link the remaining, unmeasured parts of the two states. After this process, all the unmeasured parts of the two original states are now entangled, forming a larger, complex entanglement. This is similar to connecting two independent, ent,ai"
"A new measure for entanglement in multi-particle systems (multiqudits) is introduced. It assesses both local subsystem entanglement and overall system entanglement. To fully understand multiqudit entanglement, subsystem entanglements must be considered. Additionally, this measure is extended to mixed states of multiqubits and higher-dimensional systems.,ai"

)

responses = []
total_samples = len(df)
total_batches = (total_samples + batch_size - 1) // batch_size

print(f'Processing {total_samples} samples in {total_batches} batches...')

for batch_idx in range(total_batches):
    start, end = batch_idx * batch_size, min((batch_idx + 1) * batch_size, total_samples)

    batch_texts = '\n\n'.join([f'Text {row.ID}: {row.Text}' for row in df.iloc[start:end].itertuples(index=False)])

    user_query = f'Classify the following texts as Human or AI:\n\n{batch_texts}\n\nReturn the result as a CSV with "ID" and "Label". Ensure the IDs match exactly.Make sure It is just the ID and Label, dont add anything else'

    max_retries, delay = 10, 5
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': classification_prompt},
                    {'role': 'user', 'content': user_query}
                ],
                max_tokens=200
            )

            csv_output = response.choices[0].message.content.strip()

            print(f'Batch {batch_idx + 1} API Response:\n{csv_output}\n')
            responses.append(f'Batch {batch_idx + 1}:\n{csv_output}\n\n')
            break
        except Exception as e:
            print(f'Attempt {attempt + 1} failed: {e}')
            if attempt < max_retries - 1:
                print(f'Retrying in {delay} seconds...')
                time.sleep(delay)
                delay *= 2
            else:
                responses.append(f'Batch {batch_idx + 1}: Error after max retries.\n\n')
                print('Max retries reached. Moving to next batch.')
    
    time.sleep(5)

output_path = 'classification_responses.txt'
with open(output_path, 'w', encoding='utf-8') as file:
    file.writelines(responses)

print(f'Classification completed. Raw responses saved to {output_path}')


Processing 100 samples in 10 batches...
Batch 1 API Response:
D2-100,Human
D2-11,Human
D2-12,Human
D2-13,Human
D2-15,Human
D2-16,Human
D2-17,Human
D2-18,Human
D2-2,Human
D2-21,Human

Batch 2 API Response:
ID,Label
D2-22,Human
D2-23,Human
D2-24,Human
D2-25,Human
D2-26,Human
D2-27,Human
D2-3,Human
D2-30,Human
D2-31,Human
D2-37,Human

Batch 3 API Response:
ID,Label
D2-39,Human
D2-4,Human
D2-41,Human
D2-42,Human
D2-46,Human
D2-49,Human
D2-5,Human
D2-52,Human
D2-53,Human
D2-54,AI

Batch 4 API Response:
ID,Label
D2-55,Human
D2-6,Human
D2-61,Human
D2-63,Human
D2-64,Human
D2-67,Human
D2-69,Human
D2-7,Human
D2-70,Human
D2-73,Human

Batch 5 API Response:
ID,Label
D2-77,Human
D2-78,Human
D2-80,Human
D2-82,Human
D2-85,Human
D2-88,Human
D2-9,Human
D2-90,Human
D2-92,Human
D2-95,Human

Batch 6 API Response:
ID,Label
D2-1,Human
D2-10,Human
D2-14,Human
D2-19,Human
D2-20,Human
D2-28,Human
D2-29,Human
D2-32,Human
D2-33,Human
D2-34,Human

Batch 7 API Response:
ID,Label
D2-35,Human
D2-36,Human
D2-38,Human
